<a href="https://www.kaggle.com/code/chibuikennadozie/classical-machine-learning-baseline?scriptVersionId=367134221" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/chibuikennadozie/cleaned-data/__notebook__.ipynb
/kaggle/input/notebooks/chibuikennadozie/cleaned-data/__output__.json
/kaggle/input/notebooks/chibuikennadozie/cleaned-data/__results__.html
/kaggle/input/notebooks/chibuikennadozie/cleaned-data/custom.css


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('/kaggle/input/notebooks/chibuikennadozie/cleaned-data/clean_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/notebooks/chibuikennadozie/cleaned-data/clean_data.csv'

In [ ]:
X = df.drop(columns=['diagnosis']).values
y = df['diagnosis'].values


In [ ]:
# ── Cross-validation setup ────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (RBF)'          : SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    auc  = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    f1   = cross_val_score(model, X, y, cv=cv, scoring='f1')
    acc  = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    
    results[name] = {
        'AUC_mean': auc.mean(),  'AUC_std': auc.std(),
        'F1_mean' : f1.mean(),   'F1_std' : f1.std(),
        'Acc_mean': acc.mean(),  'Acc_std': acc.std(),
    }
    print(f"{name}: AUC={auc.mean():.4f}±{auc.std():.4f}  F1={f1.mean():.4f}")

scores_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'model'})
scores_df['source'] = 'Classical ML (30 features)'

In [ ]:
# ── Visualise ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
models_list = list(results.keys())
auc_vals = [results[m]['AUC_mean'] for m in models_list]
auc_stds = [results[m]['AUC_std']  for m in models_list]
ax.barh(models_list, auc_vals, xerr=auc_stds, color='steelblue', alpha=0.8)
ax.set_xlabel('AUC (5-fold CV)')
ax.set_title('Classical ML — Baseline AUC')
ax.set_xlim(0.9, 1.01)
plt.tight_layout()
plt.savefig('baseline_auc.png', dpi=150)
plt.show()

In [ ]:
scores_df.to_csv('/kaggle/working/baseline_scores.csv', index=False)
print("Saved: baseline_scores.csv")